In [102]:
import importlib

import impish_stack
import matplotlib.pyplot as plt
import numpy as np
from astropy import units as u
from astropy.visualization import quantity_support

from adetsim import detection

importlib.reload(detection)

%matplotlib qt
plt.style.use("nice.mplstyle")

In [ ]:
lyso_energies = [60, 31, 122] << u.keV
lyso_fwhms = [30, 50, 21] << u.percent

yap_energies = [31, 60, 122] << u.keV
yap_fwhms = [27.5, 18, 13] << u.percent

energies = yap_energies
fwhms = yap_fwhms

fwhm_error = fwhms * 0.05

ereln = detection.SqrtEnergyResolution(
    reference_fwhms=fwhms, fwhm_errors=fwhm_error, reference_energies=energies
)

In [ ]:
fig, ax = plt.subplots()

with quantity_support():
    ax.errorbar(
        energies,
        fwhms,
        yerr=fwhm_error,
        color="red",
        label="FWHMs from data",
        marker=".",
        ms=12,
        capsize=6,
        ls="None",
        zorder=-1,
    )

energy_range = np.linspace(15, 200, 300)
ax.plot(
    energy_range,
    100 * ereln.resolution_function(energy_range),
    label=r"$1 / \sqrt{E}$ fit",
    color="black",
)

ax.legend()
ax.set(
    ylabel="FWHM energy resolution (%)",
    xlabel="Energy (keV)",
    title="Fitting YAP resolutions",
)

plt.show()

In [133]:
resolution_matrix = ereln.generate_resolution_matrix(energy_range << u.keV, cut=0)

In [ ]:
import matplotlib.colors as mcol

# norm = mcol.SymLogNorm(1e-5, vmin=0, vmax=1)
norm = None

fig, ax = plt.subplots()
pcm = ax.pcolormesh(energy_range, energy_range, resolution_matrix, norm=norm)
ax.set(
    xlabel="input energy bins",
    ylabel="output enregy bins",
    title="energy resolution matrix",
)
_ = fig.colorbar(pcm, label="probability")
plt.show()

In [112]:
fig, ax = plt.subplots()
for idx in range(0, resolution_matrix.shape[1], 50):
    column = resolution_matrix[:, idx]
    ax.stairs(column, energy_range)
plt.show()